# Cell tracking — working notebookEvery code cell below is runnable as-is. Cells are tagged **[CPU]** or **[GPU]**in their heading; set the accelerator accordingly before running a GPU section.Order matters only within a section. Run cell 1 first, always.

## 1. Version check — [CPU], always run first

Kaggle Dataset caching serves stale code **without raising anything**. All threemust print `2026-08-18a`. A mismatch means the Dataset was not updated — it isnot a bug in the code.

In [ ]:
import importlib, sysCODE = "/kaggle/input/datasets/faustine649/cell-tracking/cell tracking"if CODE not in sys.path:    sys.path.insert(0, CODE)import viz_cells, cell_count, submit_baseline as sbfor m in (viz_cells, cell_count, sb):    importlib.reload(m)    print(m.__name__, m.SCRIPT_VERSION)

## 2. Re-baseline the local scorer — [CPU], ~2 min

`score_one` was rewritten on 2026-08-18 against the real`tracking_cellmot/metrics.py`. The old false-positive rule required **both**endpoints of a predicted edge to match ground truth before the edge could countas an FP; the official rule needs only **one**.Every local number produced before today is optimistic by an unknown amount.Watch the line directly under `edge TP/FP/FN` — it reports how many FP the oldscorer was throwing away.

In [ ]:
sb.run_score(limit=24, max_frames=15)

## 3. Re-sweep `ratio` — [CPU], ~5 min

`--ratio 2.0` (over-detect to 2x the estimated cell count) was chosen with thebroken scorer, which could not see the false positives that over-detectioncreates. If the optimum moves down, the submitted 0.681 was tuned against ametric that does not exist.

In [ ]:
sb.run_sweep(param="ratio", values=[0.8, 1.0, 1.4, 2.0, 2.8],             limit=24, max_frames=15)

---## 4. The bare official pipeline — [GPU]

No DeepCenter, no second seed, none of the `BIOHUB_*` post-processing. Just theofficial repo plus the public weights.**Turn internet ON** (Settings -> Internet) for this section. Run 4a once persession, ~3 min.

### 4a. Install

In [ ]:
!git clone -q https://github.com/royerlab/kaggle-cell-tracking-competition.git /kaggle/working/repo!pip install -q -e /kaggle/working/repo!pip install -q "tracksdata @ git+https://github.com/royerlab/tracksdata@main" zarr tqdm

### 4b. Paths and the held-out list

Evaluate only on `6bba_` datasets. DeepCenter's `split_manifest.json` proves thewhole `6bba` batch was held out of **its** training — that is a fact, unlike ourderived 19-dataset holdout. Twelve datasets is roughly 1 GPU hour.

In [ ]:
import json, pathlibREPO    = pathlib.Path("/kaggle/working/repo")TRAIN   = pathlib.Path("/kaggle/input/competitions/"                       "biohub-cell-tracking-during-development/train")WEIGHTS = pathlib.Path("/kaggle/input/datasets/pilkwang/"                       "biohub-tracking-support-pack-50ep-v1/"                       "weights/unet_transformer/split_0/edge_predictor_best.pth")assert TRAIN.exists(),   TRAINassert WEIGHTS.exists(), WEIGHTSHOLDOUT_6BBA = ['6bba_2312ac41', '6bba_268e1230', '6bba_283bf9f1', '6bba_3a1849c2',                '6bba_3abfe10a', '6bba_5c824876', '6bba_61dd1e0d', '6bba_7af54fde',                '6bba_7b5d3b2c', '6bba_aeee7805', '6bba_afb141ff', '6bba_c27cba08'](REPO / "splits.json").write_text(json.dumps([{"train": [], "test": HOLDOUT_6BBA}]))print("splits written:", len(HOLDOUT_6BBA), "datasets")

### 4c. Smoke test on ONE dataset — ~5 min

Confirms the weights load, the geff writes, and `--evaluate` prints the officialmetric — before spending an hour on twelve.The path has **no `.zarr` suffix** on purpose: `predict()` uses the path's namefor the output filename, and `x.zarr.geff` would not match the ground truth`x.geff`.`--det-threshold 0.99` is the repo default. The 0.916 notebook uses 0.96875;that is one of the few knobs worth sweeping later.

In [ ]:
%cd /kaggle/working/repo!python scripts/predict_unet_transformer.py \    --debug-video {TRAIN}/6bba_2312ac41 \    --weights {WEIGHTS} \    --det-threshold 0.99 \    --evaluate

### 4d. The real run — 12 datasets, ~1 h

**This is THE baseline number.** Everything after it is measured as a deltaagainst it.`--evaluate` prints, per dataset and then overall:```edge TP/FP/FN, div TP/FP/FN, n_predscore = adj_edge_jaccard + 0.1 * division_jaccard   <- LB-comparable```The public leaderboard is 0.916 **with** all the extras. The gap between thisnumber and 0.916 is exactly what the post-processing pile is worth.

In [ ]:
%cd /kaggle/working/repo!python scripts/predict_unet_transformer.py \    --splits /kaggle/working/repo/splits.json --split 0 \    --weights {WEIGHTS} \    --det-threshold 0.99 \    --evaluate

## 5. Cross-check the two scorers — [CPU], ~2 min

`score_one` is our reimplementation; the repo's is the real thing. Running bothon the **same** predicted graphs is the only way to know our fast local scorercan be trusted for sweeps.`adj_edge_jaccard` from both should agree to ~0.001. If it does not, our scoreris still wrong and no local sweep means anything.One known difference remains even if the edge term matches: our`division_jaccard` is `score_divisions_simple`, a lenient approximation. Therepo's `division_metrics.py` is 574 lines of local-topology validation we havenot ported. **Trust the repo's division number, not ours.**

In [ ]:
import sys, pathlibsys.path.insert(0, "/kaggle/working/repo/scripts")from geffs_to_csv import geffs_to_csvPRED = next(pathlib.Path("/kaggle/working/repo/predictions")            .glob("*/unet_transformer/split_0"))geffs_to_csv(PRED, "/kaggle/working/official_pred.csv")sb.run_verify("/kaggle/working/official_pred.csv", gt_split="train")

---## 6. Extras — [CPU]

### Look at a movieDense datasets need an ROI crop — 700 cells in a 256x256 frame is unreadable.Red circles are ground-truth-annotated cells.

In [ ]:
viz_cells.run_gif(dataset="44b6_706092f0", roi="64,192,64,192",   # 699 cells/frame                  slab=9, step=2, fps=8, out="dense.gif")viz_cells.run_gif(dataset="6bba_2312ac41", roi="64,192,64,192",   # 120 cells/frame                  slab=9, step=2, fps=8, out="sparse.gif")from IPython.display import ImageImage("dense.gif")

### Inspect the published weight artefactsAlready run once. Attach `pilkwang/biohub-temporal-unet3d-seed314159-v1` andre-run to learn the second seed's split too.

In [ ]:
exec(open(f"{CODE}/check_weights.py").read())

In [ ]:
exec(open(f"{CODE}/read_gate.py").read())